# Slide Exercise 03: Graph-Based Movie Recommender

This is the refined version of `GraphCB_MovieRecommender_NodeVectors.ipynb`.

Learning objectives:
- Represent movies as graph nodes.
- Connect movies by shared genres and shared director.
- Recommend from graph neighborhoods.
- Understand where optional node embeddings fit.

Main functions used:
- `nx.Graph()`: creates an undirected graph.
- `add_node(...)` and `add_edge(...)`: add movies and content relationships.
- `nx.spring_layout(...)`: computes node positions for visualization.
- `nx.common_neighbors(...)`: finds graph-neighborhood overlap.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


Create a graph. Edge weights increase when movies share more content features.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
import pandas as pd

G = nx.Graph()
for _, movie in movies.iterrows():
    G.add_node(movie["title"], genres=set(movie["genres"].split("|")), director=movie["director"])

for a, b in combinations(movies["title"], 2):
    a_data = G.nodes[a]
    b_data = G.nodes[b]
    shared_genres = a_data["genres"] & b_data["genres"]
    same_director = a_data["director"] == b_data["director"]
    if shared_genres or same_director:
        G.add_edge(
            a,
            b,
            weight=len(shared_genres) + (1.5 if same_director else 0),
            reason=", ".join(sorted(shared_genres)) + ("; same director" if same_director else ""),
        )

print("Movies:", G.number_of_nodes(), "Content edges:", G.number_of_edges())


Visualize the content graph.


In [ ]:
plt.figure(figsize=(9, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_size=900, node_color="#ecfeff", edgecolors="#334155")
nx.draw_networkx_edges(G, pos, width=[G[u][v]["weight"] for u, v in G.edges()], alpha=0.45)
nx.draw_networkx_labels(G, pos, font_size=8)
plt.title("Content graph from shared genres and director")
plt.axis("off")
plt.show()


Recommend using direct graph strength plus common-neighbor overlap.


In [ ]:
def graph_score(source, target):
    direct = G[source][target]["weight"] if G.has_edge(source, target) else 0
    neighbor_overlap = len(list(nx.common_neighbors(G, source, target)))
    return direct + 0.5 * neighbor_overlap

def recommend_from_graph(title, n=5):
    rows = []
    for other in G.nodes:
        if other == title:
            continue
        rows.append({
            "input_movie": title,
            "recommended_movie": other,
            "graph_score": graph_score(title, other),
            "edge_reason": G[title][other]["reason"] if G.has_edge(title, other) else "shared graph neighborhood",
        })
    return pd.DataFrame(rows).sort_values("graph_score", ascending=False).head(n)

recommend_from_graph("Interstellar")


Optional extension:

Node2Vec or GraphSAGE can learn node embeddings from graph neighborhoods. For this course exercise, the NetworkX version is the required path because it is transparent and dependency-light.

Student task:
1. Add an edge rule for similar duration.
2. Compare graph recommendations for `Toy Story` before and after the new rule.
